# Day 09. Exercise 01
# Gridsearch

## 0. Imports

In [18]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from tqdm.notebook import tqdm
from sklearn.model_selection import cross_val_score

## 1. Preprocessing

1. Read the file [`day-of-week-not-scaled.csv`](https://drive.google.com/file/d/1AlGvsJDSzPT_70caausx8bFuupIEZkfh/view?usp=sharing). It is similar to the one from the previous exercise, but this time we did not scale continuous features (we are not going to use logreg anymore). Don't forget to enrich the table with the 'dayofweek' column from the previous day's .csv-file.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [19]:
df = pd.read_csv('../data/day-of-week-not-scaled.csv')

In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1686 entries, 0 to 1685
Data columns (total 43 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   numTrials         1686 non-null   int64  
 1   hour              1686 non-null   int64  
 2   uid_user_0        1686 non-null   float64
 3   uid_user_1        1686 non-null   float64
 4   uid_user_10       1686 non-null   float64
 5   uid_user_11       1686 non-null   float64
 6   uid_user_12       1686 non-null   float64
 7   uid_user_13       1686 non-null   float64
 8   uid_user_14       1686 non-null   float64
 9   uid_user_15       1686 non-null   float64
 10  uid_user_16       1686 non-null   float64
 11  uid_user_17       1686 non-null   float64
 12  uid_user_18       1686 non-null   float64
 13  uid_user_19       1686 non-null   float64
 14  uid_user_2        1686 non-null   float64
 15  uid_user_20       1686 non-null   float64
 16  uid_user_21       1686 non-null   float64


In [21]:
df.head()

,numTrials,hour,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,uid_user_15,...,labname_lab02,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1
0,1,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,2,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,3,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,4,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,5,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [22]:
df['dayofweek'] = pd.read_csv('../data/dayofweek.csv')['dayofweek']

In [23]:
assert len(df) == 1686

# Проверка, что столбцов 44 (43 признака + 1 целевая)
assert df.shape[1] == 44, f'Expected 44 columns, got {df.shape[1]}'

In [24]:
# Разделение на признаки и целевую переменную
X = df.drop('dayofweek', axis=1)
y = df['dayofweek']

# Разбиение на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=21,
    stratify=y
)

## 2. SVM gridsearch

1. Using `GridSearchCV` try different parameters of kernel (`linear`, `rbf`, `sigmoid`), C (`0.01`, `0.1`, `1`, `1.5`, `5`, `10`), gamma (`scale`, `auto`), class_weight (`balanced`, `None`) use `random_state=21` and `probability=True` and get the best combination of them in terms of accuracy.
2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`. Check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [25]:
# Параметры для перебора
param_grid = {
    'kernel': ['linear', 'rbf', 'sigmoid'],
    'C': [0.01, 0.1, 1, 1.5, 5, 10],
    'gamma': ['scale', 'auto'],
    'class_weight': ['balanced', None]
}

# Базовая модель
base_model = SVC(random_state=21, probability=True)

# GridSearchCV
grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

# Обучение
grid_search.fit(X_train, y_train)

# Лучшие параметры
print(f'Best parameters: {grid_search.best_params_}')
print(f'Best CV score: {grid_search.best_score_:.5f}')


# Таблица с результатами
results_df = pd.DataFrame(grid_search.cv_results_)
results_df = results_df.sort_values('rank_test_score')

# Выбор и переименование колонок
display_cols = ['rank_test_score', 'mean_test_score', 'std_test_score', 
                'param_kernel', 'param_C', 'param_gamma', 'param_class_weight']

results_display = results_df[display_cols].rename(columns={
    'rank_test_score': 'Rank',
    'mean_test_score': 'CV Accuracy',
    'std_test_score': 'Std Accuracy',
    'param_kernel': 'Kernel',
    'param_C': 'C',
    'param_gamma': 'Gamma',
    'param_class_weight': 'Class Weight'
})

# Округление для удобства чтения
results_display['CV Accuracy'] = results_display['CV Accuracy'].round(5)
results_display['Std Accuracy'] = results_display['Std Accuracy'].round(5)

# Показать топ-10 комбинаций
print('\nTop 10 combinations:')
print(results_display.head(10).to_string(index=False))

# Точность на тестовой выборке
best_model = grid_search.best_estimator_
test_accuracy = best_model.score(X_test, y_test)
print(f'\nBest model test accuracy: {test_accuracy:.5f}')

# Анализ разницы между комбинациями
top_1 = results_display.iloc[0]['CV Accuracy']
top_10 = results_display.iloc[9]['CV Accuracy']
print(f'\nDifference between Rank 1 and Rank 10: {top_1 - top_10:.5f}')

Best parameters: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf'}
Best CV score: 0.87611

Top 10 combinations:
 Rank  CV Accuracy  Std Accuracy  Kernel   C  Gamma Class Weight
    1      0.87611       0.01842     rbf  10   auto         None
    2      0.86350       0.01087     rbf  10   auto     balanced
    3      0.81602       0.00812     rbf   5   auto         None
    4      0.80861       0.02101     rbf   5   auto     balanced
    5      0.72105       0.03444  linear  10   auto     balanced
    5      0.72105       0.03444  linear  10  scale     balanced
    7      0.71959       0.01746  linear  10  scale         None
    7      0.71959       0.01746  linear  10   auto         None
    9      0.70623       0.03162  linear   5   auto     balanced
    9      0.70623       0.03162  linear   5  scale     balanced

Best model test accuracy: 0.88757

Difference between Rank 1 and Rank 10: 0.16988


## 3. Decision tree

1. Using `GridSearchCV` try different parameters of `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use `random_state=21`.
2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [26]:
# GridSearch
param_grid = {
    'max_depth': list(range(1, 50)),
    'class_weight': ['balanced', None],
    'criterion': ['entropy', 'gini']
}

base_model = DecisionTreeClassifier(random_state=21)

grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print(f'Best parameters: {grid_search.best_params_}')
print(f'Best CV score: {grid_search.best_score_:.5f}')

# Results
results_df = pd.DataFrame(grid_search.cv_results_)
results_df = results_df.sort_values('rank_test_score')

display_cols = ['rank_test_score', 'mean_test_score', 'std_test_score', 
                'param_max_depth', 'param_class_weight', 'param_criterion']

results_display = results_df[display_cols].rename(columns={
    'rank_test_score': 'Rank',
    'mean_test_score': 'CV Accuracy',
    'std_test_score': 'Std Accuracy',
    'param_max_depth': 'Max Depth',
    'param_class_weight': 'Class Weight',
    'param_criterion': 'Criterion'
})

results_display['CV Accuracy'] = results_display['CV Accuracy'].round(5)
results_display['Std Accuracy'] = results_display['Std Accuracy'].round(5)

print('\nTop 10 combinations:')
print(results_display.head(10).to_string(index=False))

best_model = grid_search.best_estimator_
test_accuracy = best_model.score(X_test, y_test)
print(f'\nBest model test accuracy: {test_accuracy:.5f}')

# Analysis
top_1 = results_display.iloc[0]['CV Accuracy']
top_10 = results_display.iloc[9]['CV Accuracy']
print(f'\nDifference between Rank 1 and Rank 10: {top_1 - top_10:.5f}')

# Check if simpler models are comparable
simple_models = results_display[results_display['Max Depth'] <= 10]
print(f'\nBest simple model (depth <= 10):')
print(simple_models.head(3).to_string(index=False))

Best parameters: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21}
Best CV score: 0.87386

Top 10 combinations:
 Rank  CV Accuracy  Std Accuracy Max Depth Class Weight Criterion
    1      0.87386       0.02507        21     balanced      gini
    2      0.87385       0.02502        25     balanced      gini
    3      0.87238       0.02526        22     balanced      gini
    4      0.87237       0.02518        49     balanced      gini
    4      0.87237       0.02518        23     balanced      gini
    4      0.87237       0.02518        27     balanced      gini
    4      0.87237       0.02518        28     balanced      gini
    4      0.87237       0.02518        29     balanced      gini
    4      0.87237       0.02518        30     balanced      gini
    4      0.87237       0.02518        31     balanced      gini

Best model test accuracy: 0.88462

Difference between Rank 1 and Rank 10: 0.00149

Best simple model (depth <= 10):
 Rank  CV Accuracy  Std Accu

## 4. Random forest

1. Using `GridSearchCV` try different parameters of `n_estimators` (`5`, `10`, `50`, `100`), `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use random_state=21.
2. Create a dataframe from the results of the gridsearch and sort it ascendengly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [27]:
# GridSearch
param_grid = {
    'n_estimators': [5, 10, 50, 100],
    'max_depth': list(range(1, 50)),
    'class_weight': ['balanced', None],
    'criterion': ['entropy', 'gini']
}

base_model = RandomForestClassifier(random_state=21)

grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print(f'Best parameters: {grid_search.best_params_}')
print(f'Best CV score: {grid_search.best_score_:.5f}')

# Results
results_df = pd.DataFrame(grid_search.cv_results_)
results_df = results_df.sort_values('rank_test_score')

display_cols = ['rank_test_score', 'mean_test_score', 'std_test_score', 
                'param_n_estimators', 'param_max_depth', 'param_class_weight', 'param_criterion']

results_display = results_df[display_cols].rename(columns={
    'rank_test_score': 'Rank',
    'mean_test_score': 'CV Accuracy',
    'std_test_score': 'Std Accuracy',
    'param_n_estimators': 'N Estimators',
    'param_max_depth': 'Max Depth',
    'param_class_weight': 'Class Weight',
    'param_criterion': 'Criterion'
})

results_display['CV Accuracy'] = results_display['CV Accuracy'].round(5)
results_display['Std Accuracy'] = results_display['Std Accuracy'].round(5)

print('\nTop 10 combinations:')
print(results_display.head(10).to_string(index=False))

best_model = grid_search.best_estimator_
test_accuracy = best_model.score(X_test, y_test)
print(f'\nBest model test accuracy: {test_accuracy:.5f}')

# Analysis
top_1 = results_display.iloc[0]['CV Accuracy']
top_10 = results_display.iloc[9]['CV Accuracy']
print(f'\nDifference between Rank 1 and Rank 10: {top_1 - top_10:.5f}')

# Check simpler models
simple_models = results_display[
    (results_display['N Estimators'] <= 50) & 
    (results_display['Max Depth'] <= 20)
]
print(f'\nBest simpler model (estimators <= 50, depth <= 20):')
print(simple_models.head(3).to_string(index=False))

Best parameters: {'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 24, 'n_estimators': 100}
Best CV score: 0.90429

Top 10 combinations:
 Rank  CV Accuracy  Std Accuracy N Estimators Max Depth Class Weight Criterion
    1      0.90429       0.01236          100        24     balanced   entropy
    2      0.90429       0.01216          100        29     balanced   entropy
    2      0.90429       0.01096           50        28         None      gini
    4      0.90355       0.01206           50        30     balanced      gini
    5      0.90355       0.01438          100        31         None      gini
    6      0.90281       0.01364          100        25     balanced   entropy
    7      0.90281       0.01363           50        33     balanced      gini
    8      0.90281       0.01046          100        45         None      gini
    8      0.90281       0.01046          100        48         None      gini
    8      0.90281       0.01046          100        47  

## 5. Progress bar

Gridsearch can be a quite long process and you may find yourself wondering when it will end.
1. Create a manual gridsearch for the same parameters values of random forest iterating through the list of the possible values and calculating `cross_val_score` for each combination. Try to increase `n_jobs`. The value `cv` for `cross_val_score` is 5.
2. Track the progress using the library `tqdm.notebook`.
3. Create a dataframe from the results of the gridsearch with the columns corresponding to the names of the parameters and `mean_accuracy` and `std_accuracy`.
4. Sort it descendingly by the `mean_accuracy`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [28]:
# Параметры для перебора
n_estimators_list = [5, 10, 50, 100]
max_depth_list = list(range(1, 50))
class_weight_list = ['balanced', None]
criterion_list = ['entropy', 'gini']

# Хранение результатов
results = []

# Общий количество комбинаций
total = len(n_estimators_list) * len(max_depth_list) * len(class_weight_list) * len(criterion_list)
print(f'Total combinations: {total}')

# Ручной перебор с прогресс-баром
for n_est in tqdm(n_estimators_list, desc='n_estimators'):
    for depth in tqdm(max_depth_list, desc='max_depth', leave=False):
        for weight in tqdm(class_weight_list, desc='class_weight', leave=False):
            for crit in tqdm(criterion_list, desc='criterion', leave=False):
                
                model = RandomForestClassifier(
                    n_estimators=n_est,
                    max_depth=depth,
                    class_weight=weight,
                    criterion=crit,
                    random_state=21,
                    # n_jobs=-1
                )
                
                scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy', n_jobs=-1)
                
                results.append({
                    'n_estimators': n_est,
                    'max_depth': depth,
                    'class_weight': weight,
                    'criterion': crit,
                    'mean_accuracy': scores.mean(),
                    'std_accuracy': scores.std()
                })



Total combinations: 784


n_estimators:   0%|          | 0/4 [00:00<?, ?it/s]

max_depth:   0%|          | 0/49 [00:00<?, ?it/s]

class_weight:   0%|          | 0/2 [00:00<?, ?it/s]

criterion:   0%|          | 0/2 [00:00<?, ?it/s]

...

criterion:   0%|          | 0/2 [00:00<?, ?it/s]

criterion:   0%|          | 0/2 [00:00<?, ?it/s]

class_weight:   0%|          | 0/2 [00:00<?, ?it/s]

criterion:   0%|          | 0/2 [00:00<?, ?it/s]

criterion:   0%|          | 0/2 [00:00<?, ?it/s]

In [29]:
# Датафрейм с результатами
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('mean_accuracy', ascending=False)
results_df['mean_accuracy'] = results_df['mean_accuracy'].round(5)
results_df['std_accuracy'] = results_df['std_accuracy'].round(5)

# Топ-10 комбинаций
print('\nTop 10 combinations:')
print(results_df.head(10).to_string(index=False))

# Анализ разницы
top_1 = results_df.iloc[0]['mean_accuracy']
top_10 = results_df.iloc[9]['mean_accuracy']
print(f'\nDifference between Rank 1 and Rank 10: {top_1 - top_10:.5f}')

# Лучшая комбинация
best = results_df.iloc[0]
print(f'\nBest parameters:')
print(f"  n_estimators: {best['n_estimators']}")
print(f"  max_depth: {best['max_depth']}")
print(f"  class_weight: {best['class_weight']}")
print(f"  criterion: {best['criterion']}")
print(f"  mean_accuracy: {best['mean_accuracy']}")


Top 10 combinations:
 n_estimators  max_depth class_weight criterion  mean_accuracy  std_accuracy
          100         24     balanced   entropy        0.90429       0.01236
           50         28         None      gini        0.90429       0.01096
          100         29     balanced   entropy        0.90429       0.01216
           50         30     balanced      gini        0.90355       0.01206
          100         31         None      gini        0.90355       0.01438
          100         25     balanced   entropy        0.90281       0.01364
           50         33     balanced      gini        0.90281       0.01363
          100         49         None      gini        0.90281       0.01046
           50         29         None      gini        0.90281       0.01170
          100         36         None      gini        0.90281       0.01046

Difference between Rank 1 and Rank 10: 0.00148

Best parameters:
  n_estimators: 100
  max_depth: 24
  class_weight: balanced
  cr

In [30]:
# результаты лучшей модели при ручном переборе полностью совпали с GridSearchCV

## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.

In [31]:
# Лучшая модель из GridSearchCV
best_model = grid_search.best_estimator_

# Предсказания на тесте
y_pred = best_model.predict(X_test)

# Финальная точность
from sklearn.metrics import accuracy_score
test_accuracy = accuracy_score(y_test, y_pred)
print(f'Final test accuracy: {test_accuracy:.5f}')

# Альтернативно (через score)
test_accuracy_alt = best_model.score(X_test, y_test)
print(f'Final test accuracy (score): {test_accuracy_alt:.5f}')

Final test accuracy: 0.92604
Final test accuracy (score): 0.92604
